# ETA baseline and models

> **All data in this notebook is FAKE (synthetic).** It comes from an invented model, not from real
> trips, patients or traffic. Nothing here shows real-world accuracy. ETAs are **estimates**, not
> guaranteed arrival times, and this project is not for real emergency dispatch.

This notebook runs top to bottom. It re-calculates the baseline result from the committed data
and reads the committed model results. Run it from VS Code with the project's `.venv` selected as
the kernel.

In [1]:
import json
import platform
import sys
from pathlib import Path

# Find the project folder (the one that contains ml/eta), wherever this notebook is opened from.
ROOT = Path.cwd().resolve()
while not (ROOT / "ml" / "eta").is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError("Open this notebook inside the eta-service project folder.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import sklearn

try:
    from IPython.display import display
except ImportError:  # plain Python
    display = print

print("project folder :", ROOT.name)
print("python         :", platform.python_version())
print("pandas         :", pd.__version__)
print("scikit-learn   :", sklearn.__version__)

project folder : eta-service
python         : 3.13.7
pandas         : 3.0.6
scikit-learn   : 1.9.1


## 1. The synthetic dataset

1000 invented trips (seed 42). Every row has `dataSource = "synthetic"`. The travel time is the
"true" time in an invented world with detours, road types, traffic and noise.

In [2]:
from ml.eta.synthetic import load_trips_csv

trips = load_trips_csv(ROOT / "data" / "synthetic_trips.csv")
print("rows:", len(trips))
assert (trips["dataSource"] == "synthetic").all(), "every row must be labelled synthetic"
print("all rows labelled synthetic: yes")
display(trips[["distanceKm", "hour", "travelTimeMinutes"]].describe().round(2))
display(trips["trafficLevel"].value_counts().rename("trips by traffic level").to_frame())
display(trips["roadType"].value_counts().rename("trips by road type").to_frame())

rows: 1000
all rows labelled synthetic: yes


,distanceKm,hour,travelTimeMinutes
count,1000.00,1000.00,1000.00
mean,11.18,11.71,27.65
std,4.61,6.96,17.18
min,0.52,0.00,1.76
25%,7.72,6.00,15.06
50%,11.50,12.00,23.68
75%,14.64,18.00,36.56
max,21.99,23.00,106.40


,trips by traffic level
trafficLevel,
low,507
medium,313
high,180


,trips by road type
roadType,
arterial,457
local,337
highway,206


## 2. The formula baseline

`distance / speed x 60`, with 25 km/h in peak hours (8 to 10 and 17 to 20) and 40 km/h otherwise.
The speeds and peak hours were accepted by the mentor but are assumptions, not measurements.

The data is split once into 800 training and 200 validation trips (seed 42). The baseline has
nothing to train, so it is scored on the 200 validation trips only.

In [3]:
from ml.eta.evaluate import (
    DEFAULT_DATA_DIR,
    baseline_predictions,
    compute_baseline_metrics,
    split_train_validation,
)

train, validation = split_train_validation(trips)
predictions = baseline_predictions(validation)
metrics = compute_baseline_metrics(validation, predictions)

saved = json.loads((DEFAULT_DATA_DIR / "baseline_metrics.json").read_text(encoding="utf-8"))
assert abs(metrics["baselineMaeMinutes"] - saved["baselineMaeMinutes"]) < 1e-4
print("train / validation trips:", len(train), "/", len(validation))
print("baseline MAE (re-calculated now):", metrics["baselineMaeMinutes"], "minutes")
print("baseline MAE (saved file)       :", saved["baselineMaeMinutes"], "minutes  -> same")
print("mean signed error:", metrics["meanSignedErrorMinutes"],
      "minutes (negative = the baseline is too low on average)")

train / validation trips: 800 / 200
baseline MAE (re-calculated now): 11.22 minutes
baseline MAE (saved file)       : 11.22 minutes  -> same
mean signed error: -8.2957 minutes (negative = the baseline is too low on average)


### 20 example predictions (randomly chosen validation trips)

In [4]:
examples = pd.read_csv(DEFAULT_DATA_DIR / "baseline_examples.csv")
print(len(examples), "examples; errorMinutes = baseline minus actual")
display(examples[["tripId", "distanceKm", "hour", "actualTravelTimeMinutes",
                  "baselineEtaMinutes", "errorMinutes"]])

20 examples; errorMinutes = baseline minus actual


,tripId,distanceKm,hour,actualTravelTimeMinutes,baselineEtaMinutes,errorMinutes
0,84,11.426,9,18.35,27.42,9.07
1,87,12.249,21,19.96,18.37,-1.59
2,129,7.243,10,22.51,17.38,-5.13
3,167,9.398,15,10.59,14.10,3.51
4,426,9.610,16,21.16,14.42,-6.74
5,468,12.681,14,47.05,19.02,-28.03
6,552,14.995,8,62.57,35.99,-26.58
7,553,6.747,20,14.41,16.19,1.78
8,630,14.575,4,21.60,21.86,0.26
9,698,7.267,0,13.88,10.90,-2.98


## 3. Linear Regression and Random Forest against the baseline

The served model is Linear Regression using only values the API receives (distance, whether the
hour is peak, and distance x peak). Random Forest is for comparison only. All numbers below are
read from `data/model_comparison.json`, which was produced by `python -m ml.eta.model`.

In [5]:
comparison = json.loads((DEFAULT_DATA_DIR / "model_comparison.json").read_text(encoding="utf-8"))
validation_results = comparison["validation"]

rows = []
for key, label in [
    ("baseline", "Formula baseline"),
    ("linearRegression", "Linear Regression (served)"),
    ("randomForest", "Random Forest (comparison)"),
    ("constantMeanReference", "Always predict the training mean"),
]:
    result = validation_results[key]
    rows.append({
        "model": label,
        "MAE all (min)": result["maeMinutes"],
        "MAE peak hours (min)": result.get("peakHours", {}).get("maeMinutes"),
        "MAE off-peak (min)": result.get("offPeakHours", {}).get("maeMinutes"),
    })
display(pd.DataFrame(rows).set_index("model"))

vs = comparison["linearRegressionVsBaseline"]
print("Baseline MAE minus Linear Regression MAE:", vs["maeDifferenceMinutes"], "minutes")
print("95% bootstrap interval:", vs["bootstrap95Interval"], " verdict:", vs["verdict"])
print("5-fold cross-validation on training trips (MAE):",
      comparison["crossValidationOnTrainRows"])
print()
print(comparison["disclaimer"])

,MAE all (min),MAE peak hours (min),MAE off-peak (min)
model,,,
Formula baseline,11.2200,12.1097,10.8203
Linear Regression (served),9.8044,12.7653,8.4741
Random Forest (comparison),9.9886,13.1658,8.5611
Always predict the training mean,12.3429,NaN,NaN


Baseline MAE minus Linear Regression MAE: 1.4156 minutes
95% bootstrap interval: [0.4299, 2.3715]  verdict: model_clearly_better
5-fold cross-validation on training trips (MAE): {'folds': 5, 'linearRegressionMaeMinutes': 8.9051, 'randomForestMaeMinutes': 9.2635}

Synthetic (fake) data only. The model learned from an invented world, so its error is NOT a measure of real-world accuracy. ETAs are estimates, not guaranteed arrival times.


**How to read this.** On this fake data the Linear Regression has a lower overall error than the
baseline, and the 95% interval for the difference is above zero. But it is **worse than the
baseline in peak hours**, and the whole comparison only says how the two rules do against the
invented world, not against real roads. The baseline is also too low on average (negative signed
error) because the invented world has detours that a straight-line speed formula ignores.

In [6]:
from ml.eta.model import load_model

loaded = load_model()
meta = loaded.metadata
print("model type          :", meta["modelType"])
print("features            :", meta["featureNames"])
print("coefficients        :", {k: round(v, 3) for k, v in meta["coefficients"].items()})
print("intercept           :", round(meta["intercept"], 3))
print("trained distance km :", {k: round(v, 2) for k, v in meta["trainingDistanceKm"].items()})
print("training rows       :", meta["trainRows"])

model type          : LinearRegression
features            : ['distanceKm', 'isPeak', 'distanceKmXPeak']
coefficients        : {'distanceKm': 2.151, 'isPeak': -2.515, 'distanceKmXPeak': 0.803}
intercept           : 1.532
trained distance km : {'min': 0.52, 'max': 21.99}
training rows       : 800


## 4. The estimator: model first, baseline as the fallback

`EtaEstimator` uses the model when it can. It uses the baseline when the model file cannot be
loaded, when the trip distance is outside the range the model was trained on (see above), or when
the model fails. Bad request values are never hidden by the fallback.

In [7]:
from ml.eta.estimator import EtaEstimator
from ml.eta.eta import predict_eta

estimator = EtaEstimator.from_model_dir()
cases = {
    "normal trip, 14:00 (about 19.8 km)": (28.6139, 77.2090, 28.5355, 77.3910, 14, "BLS"),
    "same trip at peak hour 9":            (28.6139, 77.2090, 28.5355, 77.3910, 9, "BLS"),
    "very short trip (about 0.1 km)":      (28.6139, 77.2090, 28.6148, 77.2090, 14, "BLS"),
    "very long trip (about 50 km)":        (28.6139, 77.2090, 28.9000, 77.6000, 14, "BLS"),
    "identical start and end":             (28.6139, 77.2090, 28.6139, 77.2090, 14, "BLS"),
}
table = []
for label, args in cases.items():
    outcome = estimator.estimate(*args)
    table.append({
        "case": label,
        "answered by": outcome.source,
        "why baseline": outcome.fallbackReason or "",
        "etaMinutes": outcome.result["etaMinutes"],
        "distanceKm": outcome.result["distanceKm"],
        "baseline etaMinutes": predict_eta(*args)["etaMinutes"],
    })
display(pd.DataFrame(table).set_index("case"))

,answered by,why baseline,etaMinutes,distanceKm,baseline etaMinutes
case,,,,,
"normal trip, 14:00 (about 19.8 km)",model,,44.11,19.80,29.69
same trip at peak hour 9,model,,57.48,19.80,47.51
very short trip (about 0.1 km),baseline,distance_outside_training_range,0.15,0.10,0.15
very long trip (about 50 km),baseline,distance_outside_training_range,74.47,49.65,74.47
identical start and end,baseline,distance_outside_training_range,0.00,0.00,0.00


In [8]:
from ml.eta.validation import InvalidInputError

bad_requests = {
    "hour sent as 8.0":        (28.6139, 77.2090, 28.5355, 77.3910, 8.0, "BLS"),
    "hour sent as text '8'":   (28.6139, 77.2090, 28.5355, 77.3910, "8", "BLS"),
    "latitude 95":             (95, 77.2090, 28.5355, 77.3910, 14, "BLS"),
    "ambulance type 'bls'":    (28.6139, 77.2090, 28.5355, 77.3910, 14, "bls"),
    "ambulance type 'ALS'":    (28.6139, 77.2090, 28.5355, 77.3910, 14, "ALS"),
}
for label, args in bad_requests.items():
    try:
        estimator.estimate(*args)
        print(f"{label}: ACCEPTED (unexpected)")
    except InvalidInputError as error:
        print(f"{label}: rejected -> {error}")

hour sent as 8.0: rejected -> hour must be a whole number (integer).
hour sent as text '8': rejected -> hour must be a whole number (integer).
latitude 95: rejected -> fromLat must be between -90 and 90.
ambulance type 'bls': rejected -> ambulanceType must be one of: BLS.
ambulance type 'ALS': rejected -> ambulanceType must be one of: BLS.


## 5. The HTTP API (sample request and response)

This uses FastAPI's test client, so no server needs to be running. To run a real server:
`uvicorn api.main:app --host 127.0.0.1 --port 8000`.

In [9]:
try:
    from fastapi.testclient import TestClient
except ImportError:
    print("fastapi/httpx are not installed in this environment; skipping this cell.")
else:
    from api.main import create_app

    request_body = {"fromLat": 28.6139, "fromLng": 77.2090, "toLat": 28.5355,
                    "toLng": 77.3910, "hour": 14, "ambulanceType": "BLS"}
    with TestClient(create_app()) as client:
        good = client.post("/eta", json=request_body)
        print("request :", request_body)
        print("status  :", good.status_code, "| X-ETA-Source:", good.headers["x-eta-source"])
        print("response:", good.json())
        print()
        bad = client.post("/eta", json={**request_body, "hour": "8"})
        print("bad request (hour as text) status:", bad.status_code)
        print("response:", bad.json())
        print()
        print("health  :", client.get("/health").json())

request : {'fromLat': 28.6139, 'fromLng': 77.209, 'toLat': 28.5355, 'toLng': 77.391, 'hour': 14, 'ambulanceType': 'BLS'}
status  : 200 | X-ETA-Source: model
response: {'etaMinutes': 44.11, 'distanceKm': 19.8}

bad request (hour as text) status: 422
response: {'detail': [{'loc': ['body', 'hour'], 'msg': 'hour must be a whole number (integer).', 'type': 'value_error'}]}

health  : {'status': 'ok', 'modelLoaded': True}


c:\Users\seden\eta-service\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## Limits to remember

* Everything above is on **fake data**. It is not a validation against real traffic.
* The baseline speeds and peak hours are accepted assumptions, not measurements.
* The model only answers for trips between about 0.52 and 21.99 km (what it was trained on). Outside
  that range the baseline answers, and the two can differ by 11 to 16 minutes at the upper edge.
* More in `docs/limitations_and_blockers.md` and `docs/test_plan.md`.